In [1]:
import os#, sys, csv, argparse
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

from utils.dataclass import TimeSeriesInferenceDataset
from utils.load_and_clean_real_data import load_and_clean_real_data

from utils.mcmc import logprior, loglike_alpha_mu_omega, next_MCMC_sample


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)


In [3]:
grid_estimation = torch.tensor((0.04,1,0.615)).to(device)
jumpsize_proposal = torch.tensor((1e-2,1e-2,1e-2)).to(device)


In [4]:
real_data_path = "real_data/Exp_1_live_lowpH.csv"

filename = os.path.basename(real_data_path)  # Exp_1_live_lowpH.csv
mcmc_output_path = os.path.join("mcmc_results", filename)

os.makedirs("mcmc_results", exist_ok=True)


In [5]:
df = load_and_clean_real_data(real_data_path, cutoff=300)
ts     = torch.tensor(df["Day"].values * 24)
counts = torch.tensor(df["Counts"].values)
dils   = torch.tensor(df["Dilution"].values)

kappa_np = np.load("kappa_samples_Exp1_4gauss.npz")["kappa_samples"]
kappa_np.sort()
print((kappa_np>1000).mean())
kappa_np = kappa_np[kappa_np>1000]

dataset = TimeSeriesInferenceDataset(
    ts=ts, counts=counts, dils=dils,
    kappa_samples=kappa_np,
    cutoff=300,
    t_switch=None,
    rho=1.,
    device=device,
)


[WARN] Missing columns skipped: CFU_2, CFU_2222
[INFO] Using dilutions: [22, 222]
[INFO] Loaded 350 rows from real_data/Exp_1_live_lowpH.csv
[INFO] Retained 349 rows with valid CFU entries.
[INFO] Dropped 1 rows with no valid CFU values at any dilution.
0.9876708984375
Dataset loaded: 349 points, 5 timepoints.


In [6]:
N_samples=1000

samples = []
lps = []

logposterior = lambda th: loglike_alpha_mu_omega(dataset, *th) + 0*logprior(*th)

start = grid_estimation.clone()
theta = start
lp = logposterior(theta)
print(lp)


tensor([-1959.5388], device='cuda:0')


/home/pessoa/Codes/Celegans_inference/utils/mcmc.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  - torch.lgamma(torch.tensor(k, dtype=x.dtype, device=x.device))


In [ ]:
for i in tqdm(range(N_samples)):
    lp = logposterior(theta)
    theta, lp = next_MCMC_sample(logposterior, theta, lp, jumpsize_proposal)
    samples.append(theta.clone().cpu().numpy())
    lps.append(lp.item()*1.)
    

    if (i + 1) % 10 == 0:
        samples_tensor = np.vstack(samples)

        df = pd.DataFrame({
            "alpha": samples_tensor[:, 0],
            "mu": samples_tensor[:, 1],
            "omega": samples_tensor[:, 2],
            "logposterior": np.array(lps)
        })

        df.to_csv(mcmc_output_path)

        print(f"Saved at iteration {i+1}")
        print (theta,lp.item())


  1%|          | 10/1000 [03:23<5:37:34, 20.46s/it]

Saved at iteration 10
tensor([0.0397, 1.0189, 0.6235], device='cuda:0') -1957.15576171875


  2%|▏         | 20/1000 [07:11<6:14:13, 22.91s/it]

Saved at iteration 20
tensor([0.0413, 1.0222, 0.6168], device='cuda:0') -1959.6624755859375


  3%|▎         | 30/1000 [11:07<6:32:00, 24.25s/it]

Saved at iteration 30
tensor([0.0414, 1.0547, 0.6462], device='cuda:0') -1950.258056640625


  4%|▎         | 37/1000 [13:56<6:23:04, 23.87s/it]